# VLA02：多速率执行、插值与规则式接触调节

保持相同1 kHz输出网格，独立配置sensor、context和regulator频率；用简化spring contact解释feedback与stale context。

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)


## 2. 声明所有时钟与简化物理假设
不是FP2模型或Flexiv控制器。x理想跟踪，只有z接触运动；无惯性/摩擦，F=K max(surface−z,0)。低频输出端点之间在1 kHz网格线性插值。


In [ ]:
OUTPUT_HZ, SENSOR_HZ, CONTEXT_HZ = 1000, 100, 2
DURATION, STIFFNESS, NOMINAL_STIFFNESS = 8., 1500., 1200.
GAIN, MAX_Z_SPEED, SENSOR_DELAY = .0015, .03, .010
dt = 1./OUTPUT_HZ
time = np.arange(int(DURATION*OUTPUT_HZ))/OUTPUT_HZ
x = .02*time
surface = .004*np.sin(2*np.pi*x/.12) + .001*(time>=5.25)
true_goal = np.where(time<4.25, 8., 12.)
sensor_stride = OUTPUT_HZ//SENSOR_HZ
context_stride = OUTPUT_HZ//CONTEXT_HZ
sensor_noise = np.random.default_rng(SEED).normal(scale=.05, size=len(time))
def goal_at(t):
    return 8. if t<4.25 else 12.
# Shared nominal RFM-like motion reference: task goal updated at 2 Hz,
# geometric waypoints at 50 Hz, same interpolated reference for all methods.
reference_time = np.arange(int(DURATION*50)+1)/50.
context_sample_time = np.floor(reference_time*CONTEXT_HZ)/CONTEXT_HZ
reference_goal = np.where(context_sample_time<4.25, 8., 12.)
reference_z = .002*np.sin(2*np.pi*.02*reference_time/.12)-reference_goal/NOMINAL_STIFFNESS
interpolated_motion = np.interp(time, reference_time, reference_z)


## 3. 同一个plant与sensor；四种执行策略
Position-only使用共享motion reference。其余在z轴使用规则式导纳，x继续相同运动。Slow/fast使用相同增益以隔离频率，不宣称它们分别经过最优调参。Stale-context仅让下游目标力保持8 N。


In [ ]:
def rollout(policy_hz=50, position_only=False, stale_context=False):
    assert OUTPUT_HZ % policy_hz == 0
    policy_stride = OUTPUT_HZ//policy_hz
    z = float(interpolated_motion[0])
    start_z = end_z = z; last_policy_step = 0
    measured = 0.; cached_goal = goal_at(0.)
    force_log = np.zeros(len(time)); z_log = np.zeros(len(time))
    available_samples = np.full(len(time), np.nan)
    age_log = np.full(len(time), np.nan)
    policy_updates = sensor_updates = context_updates = 0
    for index, timestamp in enumerate(time):
        actual_force = STIFFNESS*max(surface[index]-z, 0.)
        force_log[index] = actual_force; z_log[index] = z
        if index % sensor_stride == 0:
            available_samples[index] = actual_force+sensor_noise[index]
            sensor_updates += 1
        source_index = int(np.floor((timestamp-SENSOR_DELAY)*SENSOR_HZ+1e-9))*sensor_stride
        if source_index >= 0:
            assert source_index <= index and np.isfinite(available_samples[source_index])
            measured = float(available_samples[source_index])
            age_log[index] = timestamp-source_index/OUTPUT_HZ
        if index % context_stride == 0:
            if not stale_context:
                cached_goal = goal_at(timestamp)
            context_updates += 1
        if position_only:
            z = float(interpolated_motion[min(index+1, len(time)-1)])
        else:
            if index % policy_stride == 0:
                velocity = float(np.clip(-GAIN*(cached_goal-measured), -MAX_Z_SPEED, MAX_Z_SPEED))
                start_z, end_z = z, z+velocity/policy_hz
                last_policy_step = index; policy_updates += 1
            fraction = min((index-last_policy_step+1)/policy_stride, 1.)
            z = (1-fraction)*start_z+fraction*end_z
    return {'force': force_log, 'z': z_log, 'age': age_log,
            'policy_updates': policy_updates, 'sensor_updates': sensor_updates,
            'context_updates': context_updates}
traces = {
    'position only': rollout(position_only=True),
    '2 Hz force regulator': rollout(policy_hz=2),
    '50 Hz force regulator': rollout(policy_hz=50),
    '50 Hz, stale force context': rollout(policy_hz=50, stale_context=True),
}


## 4. 评价力误差、contact loss与新信息次数
统一排除最初0.5s启动段；sensor在该段之前无历史测量，以0初始化。这里的1 kHz只表示共同输出网格，不表示1 kHz新sensor/learned predictions。


In [ ]:
evaluation = time>=.5; rows = []
for name, trace in traces.items():
    error = trace['force'][evaluation]-true_goal[evaluation]
    rows.append({'method': name, 'force_MAE_N': float(np.abs(error).mean()),
                 'force_peak_N': float(trace['force'].max()),
                 'contact_loss_fraction': float((trace['force'][evaluation]<.5).mean()),
                 'rule_updates': trace['policy_updates'], 'new_sensor_samples': trace['sensor_updates'],
                 'context_checks': trace['context_updates'], 'output_samples': len(time)})
display(pd.DataFrame(rows))
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(time, true_goal, '--', color='black', label='desired force')
for name, trace in traces.items():
    axes[0].plot(time, trace['force'], label=name, alpha=.8)
axes[0].set(ylabel='force (N)', title='Shared output grid, different feedback/context updates')
axes[0].legend()
axes[1].plot(time, surface, '--', color='black', label='surface')
for name, trace in traces.items():
    axes[1].plot(time, trace['z'], label=name, alpha=.8)
axes[1].set(xlabel='time (s)', ylabel='z (m)'); plt.tight_layout(); plt.show()
results = {'seed': SEED, 'output_hz': OUTPUT_HZ, 'sensor_hz': SENSOR_HZ,
           'context_hz': CONTEXT_HZ, 'sensor_delay_seconds': SENSOR_DELAY,
           'excluded_startup_seconds': .5, 'metrics': rows,
           'scope': 'Rule-based regulator and simplified spring contact; no learned FP2 or robot-performance validation.'}


## 面试追问与实验记录
1. 输出8000个点是否意味着8000次策略决策？
2. 把sensor降到25 Hz，为什么提高输出插值率不能消除sensor age？
3. 为什么stale context可能给出稳定但不正确的目标力？
4. 同增益频率对比与各自最优调参比较有何不同？
5. 加入惯性、摩擦或旋转后，当前实验哪些结论需重新验证？

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
